In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionWithCache(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, cache=None):
        # token_ids: (B, T) - long tensor of ids
        x = self.embedding(token_ids)  # (B, T, d_model)

        q = self.wq(x)  # (B, T, d_model)
        k = self.wk(x)
        v = self.wv(x)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=1)  # (B, T_total, d_model)
            v = torch.cat([past_v, v], dim=1)

        new_cache = (k, v)

        scores = q @ k.transpose(-2, -1)  # (B, T, T_total)
        scores = scores / (k.shape[-1] ** 0.5)

        if q.shape[1] > 1:
            T_q, T_k = q.shape[1], k.shape[1]
            mask = torch.triu(torch.ones(T_q, T_k), diagonal=T_k - T_q + 1).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ v  # (B, T, d_model)

        logits = self.lm_head(out)  # (B, T, vocab_size)

        return logits, new_cache


@torch.no_grad()
def generate(model, token_ids, max_new_tokens):
    cache = None
    generated = token_ids

    for _ in range(max_new_tokens):
        if cache:
            print(f'KV Cache shape: K {cache[0].shape} V: {cache[1].shape}, turn {_}')
        else:
            print(f'Actual KV Cache shape: {cache}, turn {_}')
        print(f'Turn {_} and the shape of tokens is: {generated.shape}')
        print(f'=' * 50)

        model_input = generated if cache is None else generated[:, -1:]

        logits, cache = model(model_input, cache=cache)

        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # (B, 1)
        generated = torch.cat([generated, next_token], dim=1)
    return generated


# uso
vocab_size = 50
model = AttentionWithCache(d_model=16, vocab_size=vocab_size)

prompt = torch.tensor([[1, 2, 3]], dtype=torch.long)  # (1, 3)
print(f'Actual shape of input (prompt): {prompt.shape}')
print(f'=' * 50)

output = generate(model, prompt, max_new_tokens=10)
print(f'Final shape of input (prompt): {output.shape}')

Actual shape of input (prompt): torch.Size([1, 3])
Actual KV Cache shape: None, turn 0
Turn 0 and the shape of tokens is: torch.Size([1, 3])
KV Cache shape: K torch.Size([1, 3, 16]) V: torch.Size([1, 3, 16]), turn 1
Turn 1 and the shape of tokens is: torch.Size([1, 4])
KV Cache shape: K torch.Size([1, 4, 16]) V: torch.Size([1, 4, 16]), turn 2
Turn 2 and the shape of tokens is: torch.Size([1, 5])
KV Cache shape: K torch.Size([1, 5, 16]) V: torch.Size([1, 5, 16]), turn 3
Turn 3 and the shape of tokens is: torch.Size([1, 6])
KV Cache shape: K torch.Size([1, 6, 16]) V: torch.Size([1, 6, 16]), turn 4
Turn 4 and the shape of tokens is: torch.Size([1, 7])
KV Cache shape: K torch.Size([1, 7, 16]) V: torch.Size([1, 7, 16]), turn 5
Turn 5 and the shape of tokens is: torch.Size([1, 8])
KV Cache shape: K torch.Size([1, 8, 16]) V: torch.Size([1, 8, 16]), turn 6
Turn 6 and the shape of tokens is: torch.Size([1, 9])
KV Cache shape: K torch.Size([1, 9, 16]) V: torch.Size([1, 9, 16]), turn 7
Turn 7 and